# Temporal decorrelation of speckle / ghost images

How fast does the speckle pattern decorrelate in time, and how does the **integrator** (`int`) compare with the **predictive** (`pred`) controller? The analysis works on pre-processed stacks (`.npy`, shape `(T, Ny, Nx)`).

**Contents**

1. Setup and configuration
2. Function library (all reusable code lives here)
3. Data inspection: raw frames, batches, outliers
4. Regions and masks
5. Frame-to-frame correlation matrices
6. Lifetime estimation, estimator by estimator
7. Comparing runs (integrator vs predictive)
8. Stored results (`temporal_correlation.h5`)
9. Appendix: archived drafts

**Conventions**

- A stack is `(T, Ny, Nx)` with time on axis 0; `dt` is the time between frames in seconds.
- A *lifetime* is the lag at which a pixel's temporal autocorrelation (ACF) drops to a threshold: 0.5 is the half-correlation time, 0.1 is the $t_{90}$ (90 % decorrelated).
- Run files are named `<algo>_<ro>ro_<ws>ws[_<key>][_<count>].npy`, handled by `run_name` / `load_stack`.

## 1. Setup and configuration

### 1.1 Imports

In [ ]:
import glob
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import tifffile as tiff
from astropy.io import fits

# scientific libraries
from applefy.utils.positions import center_subpixel
from photutils.aperture import CircularAnnulus, CircularAperture
from scipy.optimize import curve_fit

# plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# project modules (provide e.g. `titling`, `reorder_name`).
# Imported *before* the function library so the definitions below take precedence.
from utils.statistics import *
from utils.naming import *
from pipeline_pre_ADI.pre_process.pre_processing import *
from pipeline_pre_ADI.pre_process.plot_pre_processing import *
from pipeline_pre_ADI.pre_process.correlation import *

mpl.rcParams["hatch.linewidth"] = 0.5
plt.close("all")

### 1.2 Configuration

Paths and constants are defined **only here**; every later cell refers to these names.

In [ ]:
# --- Data locations ----------------------------------------------------------
GHOST = Path("/home/aosimul/noah/data/ghost_images")
RESULTS = Path("/home/aosimul/noah/src/pipeline_pre_ADI/results")

RAW_TIF_DIR = GHOST / "phase_screens_SLM/8_24"      # raw .tif frames
RUNS_10_15 = GHOST / "10_15_20ws"                   # batches _1 / _2, DIT = 10 ms
LONG_RUN = GHOST / "long_7ro_20ws"                  # stacked _1-4 / _5-8, DIT = 2.814 ms
LONG_RUN_FOCUS = GHOST / "long_7ro_20ws_focus"      # same, with focus
RESULTS_10_15 = RESULTS / "10_15_20ws"              # temporal_correlation.h5
SPECKLE_PNG_DIR = RESULTS_10_15 / "speckle_diversity"

# --- Acquisition: time between frames [s] -------------------------------------
DIT_PSF = 29e-6
DIT_10MS = 10e-3
DIT_LONG = 2.814e-3

# --- Geometry ------------------------------------------------------------------
RADIUS = 55                   # crop radius [px]
MASK_RADIUS = 35              # circular analysis mask [px]
IMG_SHAPE = (111, 111)
MASK_APERTURE = CircularAperture((RADIUS, RADIUS), MASK_RADIUS)
MASK_OUTLIER = MASK_APERTURE.to_mask(method="center")

# Control regions: radii [px] from the image centre
REGION_EDGES = dict(edge_coro=10, edge_stg1=34, edge_stg2=60, edge_atm=160)

# --- Experiment grid -----------------------------------------------------------
ALGOS = {"int": ("Integrator", "tab:blue"), "pred": ("Predictive", "tab:orange")}
ROS = [5, 7, 10]
WSS = [10, 15, 20]

## 2. Function library

All reusable code is collected in this section. Run these cells once; sections 3-8 only *call* them.

### 2.1 Data access

In [ ]:
# Included in utils.naming

# def run_name(ro, ws, key=None, count=None):
#     """Run tag used in file names, e.g. '7ro_20ws_1key_1-4'."""
#     parts = [f"{ro}ro_{ws}ws"]
#     parts += [str(p) for p in (key, count) if p is not None]
#     return "_".join(parts)


# def iter_runs(ros, wss, keys=(None,), counts=(None,)):
#     """Yield one dict(ro, ws, key, count) per combination."""
#     for count in counts:
#         for key in keys:
#             for ro in ros:
#                 for ws in wss:
#                     yield dict(ro=ro, ws=ws, key=key, count=count)


# def load_stack(folder, algo, dtype=None, **run):
#     """Load '<folder>/<algo>_<run_name>.npy'. `run` takes ro, ws, key, count."""
#     stack = np.load(Path(folder) / f"{algo}_{run_name(**run)}.npy")
#     return stack if dtype is None else stack.astype(dtype)


# def list_raw_files(folder):
#     """Split the .tif files of `folder` into (science, psf) lists."""
#     files = sorted(glob.glob(str(Path(folder) / "*.tif")))
#     is_psf = lambda f: "no_coro" in Path(f).name
#     science = [f for f in files if "coro" in Path(f).name and not is_psf(f)]
#     return science, [f for f in files if is_psf(f)]


# def read_tif_stack(path, n_frames=None):
#     """Read the first `n_frames` frames of a .tif file (all frames if None)."""
#     return tiff.imread(path, key=slice(0, n_frames))


# def read_raw_contrast(store, tag, level="pixel_rad"):
#     """(radius, raw_contrast) of entry '/<tag>' in an open pd.HDFStore.

#     `level` is 'pixel_rad' (pixels) or 'pixel_rad_fwhm' (units of FWHM)."""
#     df = store[f"/{tag}"]
#     return df.index.get_level_values(level), df["raw_contrast"]

### 2.2 Regions and masks

`circular_region_masks` builds the four control regions (the coronagraph mask `coro` is everything *outside* the central disk). `make_grid_masks` tiles the image into squares for spatially resolved correlation analysis.

In [ ]:
REGION_TITLES = {
    "coro": "Outside coronagraph",
    "stage_1": "Stage 1 control radius",
    "stage_2": "Stage 2 control radius",
    "rest": "Uncontrolled radius",
}


def circular_region_masks(shape, center, edge_coro=10, edge_stg1=34, edge_stg2=60,
                          edge_atm=160, overlap=2):
    """Boolean masks of the control regions around `center`.

    Rings start `overlap` px inside the previous edge so neighbouring regions overlap slightly."""
    disk = lambda r: CircularAperture(center, r).to_mask(method="center").to_image(shape).astype(bool)
    ring = lambda r_in, r_out: (
        CircularAnnulus(center, r_in, r_out).to_mask(method="center").to_image(shape).astype(bool)
    )
    return {
        "coro": ~disk(edge_coro),
        "stage_1": ring(edge_coro - overlap, edge_stg1),
        "stage_2": ring(edge_stg1 - overlap, edge_stg2),
        "rest": ring(edge_stg2 - overlap, edge_atm),
    }


def make_grid_masks(shape, grid_size=12, include_partial=False, mask_grid=None):
    """Square tiles covering an image, as {'y<y0>:<y1>_x<x0>:<x1>': bool mask}.

    include_partial : also keep incomplete tiles at the image edges.
    mask_grid       : optional boolean image; tiles are intersected with it and empty tiles dropped.
    """
    ny, nx = shape
    if mask_grid is not None:
        mask_grid = np.asarray(mask_grid, dtype=bool)
        if mask_grid.shape != tuple(shape):
            raise ValueError(f"Mask shape {mask_grid.shape} does not match image shape {tuple(shape)}")

    masks = {}
    for y in range(0, ny, grid_size):
        for x in range(0, nx, grid_size):
            y_end, x_end = min(y + grid_size, ny), min(x + grid_size, nx)
            if not include_partial and (y_end - y < grid_size or x_end - x < grid_size):
                continue
            tile = np.zeros((ny, nx), dtype=bool)
            tile[y:y_end, x:x_end] = True
            if mask_grid is not None:
                tile &= mask_grid
                if not tile.any():
                    continue
            masks[f"y{y}:{y_end}_x{x}:{x_end}"] = tile
    return masks

### 2.3 Frame-to-frame correlation

`temporal_correlation` correlates every pair of frames inside each region (pixels = samples), averages along the diagonals to get correlation versus frame lag, and optionally fits $A\,e^{-d/\xi}+C_0$ to the mean profile.

In [ ]:
def exp_decay(d, A, xi, C0):
    """Exponential decay model A * exp(-d / xi) + C0."""
    return A * np.exp(-d / xi) + C0


def frame_correlation_matrix(X):
    """Pearson correlation between all frame pairs of X (T, n_pixels).

    Only the upper triangle (k=1) is filled; everything else is NaN."""
    corr = np.corrcoef(X)
    out = np.full(corr.shape, np.nan)
    i, j = np.triu_indices(len(X), k=1)
    out[i, j] = corr[i, j]
    return out


def lag_profiles(cmat):
    """Median / mean / std of a correlation matrix along its diagonals (lag 1, 2, ...)."""
    diagonals = [np.diagonal(cmat, offset=d) for d in range(1, cmat.shape[0])]
    return {
        "median": np.array([np.nanmedian(x) for x in diagonals]),
        "mean": np.array([np.nanmean(x) for x in diagonals]),
        "std": np.array([np.nanstd(x) for x in diagonals]),
    }


def fit_exp_decay(mean, std, t_max=100, fit_sigma=False):
    """Fit `exp_decay` to the first `t_max` lags of a mean lag profile.

    Returns (params, params_std) = ((A, xi [samples], C0), ...); NaNs if the fit fails."""
    mean, std = mean[:t_max], std[:t_max]
    d = np.arange(1, len(mean) + 1)
    ok = np.isfinite(mean) & np.isfinite(std) & (std > 0)
    d, mean, std = d[ok], mean[ok], std[ok]
    p0 = [mean[0] - mean[-1], t_max / 5, mean[-1]]   # amplitude, decay length, baseline
    try:
        params, cov = curve_fit(
            exp_decay, d, mean,
            sigma=std if fit_sigma else None, absolute_sigma=bool(fit_sigma),
            p0=p0, maxfev=10000,
        )
    except RuntimeError:
        return np.full(3, np.nan), np.full(3, np.nan)
    return params, np.sqrt(np.diag(cov))


def temporal_correlation(stack, masks=None, grid_size=None, sub=None, t_max=100,
                         mask_grid=None, include_partial=False, fit=True, fit_sigma=False):
    """Frame-to-frame correlation statistics per region.

    Regions: `masks` ({name: bool image}) if given, else a square grid of `grid_size`
    pixels, else the full image (key 'full').
    sub : subtract the temporal 'median' or 'mean' of every pixel first.

    Returns {name: dict(correlation_matrix, correlation_median, correlation_1d_median,
    correlation_1d_mean, correlation_1d_std[, exp_fit_params, exp_fit_std])}.
    """
    stack = np.asarray(stack)
    if sub == "median":
        stack = stack - np.median(stack, axis=0)
    elif sub == "mean":
        stack = stack - np.mean(stack, axis=0)

    if masks is None:
        if grid_size is not None:
            masks = make_grid_masks(stack.shape[1:], grid_size, include_partial, mask_grid)
        else:
            masks = {"full": np.ones(stack.shape[1:], dtype=bool)}

    results = {}
    for name, region in masks.items():
        cmat = frame_correlation_matrix(stack[:, region])
        prof = lag_profiles(cmat)
        res = {
            "correlation_matrix": cmat,
            "correlation_median": np.nanmedian(cmat),
            "correlation_1d_median": prof["median"],
            "correlation_1d_mean": prof["mean"],
            "correlation_1d_std": prof["std"],
        }
        if fit:
            res["exp_fit_params"], res["exp_fit_std"] = fit_exp_decay(
                prof["mean"], prof["std"], t_max, fit_sigma
            )
        results[name] = res
    return results


def block_lifetime_map(stack, grid_size=4, threshold=0.1, dt=DIT_10MS):
    """Coarse lifetime map from block correlation matrices.

    Per `grid_size` x `grid_size` block: first lag at which the median correlation profile
    is closest to `threshold` * (overall median correlation). Returns an image of
    (Ny // grid_size, Nx // grid_size) blocks in seconds."""
    ny, nx = stack.shape[1:]
    out = np.full((ny // grid_size, nx // grid_size), np.nan)
    for tile in make_grid_masks((ny, nx), grid_size).values():
        ys, xs = np.nonzero(tile)
        cmat = frame_correlation_matrix(stack[:, tile])
        median_profile = lag_profiles(cmat)["median"]
        lag = np.argmin(np.abs(median_profile - threshold * np.nanmedian(cmat))) + 1
        out[ys.min() // grid_size, xs.min() // grid_size] = lag * dt
    return out

### 2.4 Lifetime estimators

All estimators map a stack `(T, Ny, Nx)` to a lifetime image `(Ny, Nx)` in seconds. They are explained one by one in section 6.

| Function | Idea | Needs |
|---|---|---|
| `lifetime_map` | half-correlation time from the PSD (Wiener-Khinchin) | stationarity |
| `crossing_time(temporal_acf(...))` | first lag where the zero-padded ACF drops below a threshold, interpolated | nothing else |
| `t90_lag1` | one lag only, assumes a single exponential | exponential decay |
| `t90_block_variance` | variance of block means, no ACF at all | blocks longer than $\tau$ |

`lifetime_from_stack` picks one by name.

In [ ]:
def temporal_psd(stack, dt):
    """One-sided temporal power spectral density of every pixel.

    Returns (freq [Hz], psd [intensity^2 / Hz]) with shapes (T//2 + 1,) and (T//2 + 1, Ny, Nx).
    The input is used as is, so the DC bin (index 0) holds the temporal mean."""
    N = stack.shape[0]
    P = np.abs(np.fft.rfft(stack, axis=0)) ** 2
    psd = (dt / N) * P                 # two-sided normalisation
    if N > 2:
        psd[1:-1] *= 2                 # fold negative frequencies (one-sided)
    return np.fft.rfftfreq(N, d=dt), psd


def half_variance_frequency(freq, pixel_psd):
    """(total variance, frequency below which half of it lies) for one pixel, DC bin excluded."""
    df = freq[1] - freq[0]
    cumulative = np.cumsum(pixel_psd[1:]) * df
    idx = np.searchsorted(cumulative, 0.5 * cumulative[-1])
    return cumulative[-1], freq[1:][idx]


def wiener_khinchin_acf(stack, subtract="median"):
    """Circular temporal ACF per pixel via the Wiener-Khinchin theorem (IFFT of the PSD).

    Normalised so that lag 0 equals 1. Returns an array shaped like `stack`."""
    x = np.asarray(stack, dtype=np.float32)
    if subtract == "median":
        x = x - np.median(x, axis=0, keepdims=True)
    elif subtract == "mean":
        x = x - np.mean(x, axis=0, keepdims=True)
    P = np.abs(np.fft.rfft(x, axis=0)) ** 2
    acf = np.fft.irfft(P, n=x.shape[0], axis=0)
    zero_lag = acf[0:1]
    return np.divide(acf, zero_lag, out=np.full_like(acf, np.nan), where=zero_lag > 0)


def temporal_acf(stack, unbiased=True):
    """Temporal ACF per pixel, (T, H, W) -> (T, H, W), lag 0 == 1.

    The trace is zero-padded to >= 2T - 1 samples before the FFT. Without the padding, lag k
    would wrap the end of the trace onto its start and the ACF would decay artificially fast.
    unbiased : divide lag k by its number of overlapping pairs (T - k)."""
    T = stack.shape[0]
    x = np.asarray(stack, dtype=np.float64) - stack.mean(axis=0, keepdims=True)
    n = 1 << (2 * T - 1).bit_length()
    F = np.fft.rfft(x, n=n, axis=0)
    ac = np.fft.irfft(F * np.conj(F), n=n, axis=0)[:T]
    if unbiased:
        ac /= np.arange(T, 0, -1, dtype=np.float32)[:, None, None]
    var = ac[0]
    return np.divide(ac, var, out=np.full_like(ac, np.nan), where=var > 0)


def crossing_time(acf, thresh=0.1, dt=DIT_10MS):
    """Time [s] at which the ACF first falls below `thresh`, linearly interpolated between
    frames (sub-frame precision). NaN where it never does."""
    below = acf < thresh
    found = below.any(axis=0)
    k = np.clip(below.argmax(axis=0), 1, None)          # first lag below threshold
    ii, jj = np.indices(acf.shape[1:])
    y_hi, y_lo = acf[k - 1, ii, jj], acf[k, ii, jj]
    frac = (y_hi - thresh) / (y_hi - y_lo)
    return np.where(found, (k - 1 + frac) * dt, np.nan)


def lifetime_map(stack, dt=DIT_10MS, thresh=0.5):
    """Half-correlation lifetime [s]: PSD -> ACF (Wiener-Khinchin) -> first crossing of `thresh`.

    Uses lags up to T // 2 only."""
    acf = wiener_khinchin_acf(stack)[: stack.shape[0] // 2]
    return crossing_time(acf, thresh=thresh, dt=dt).astype(np.float32)


def t90_lag1(stack, dt=DIT_10MS):
    """t90 [s] from the lag-1 autocorrelation r1, assuming exponential decay:
    tau = -dt / ln(r1), t90 = ln(10) * tau."""
    x = np.asarray(stack, np.float32)
    x = x - x.mean(0, keepdims=True)                    # new array: the input stays untouched
    r1 = np.einsum("thw,thw->hw", x[:-1], x[1:]) / np.einsum("thw,thw->hw", x, x)
    r1 = np.clip(r1, 1e-3, 0.999)
    return np.log(10) * (-dt / np.log(r1))


def t90_block_variance(stack, m=10, dt=DIT_10MS):
    """t90 [s] from the variance of m-frame block means: tau = (m dt / 2) Var(blocks) / Var(x)."""
    T = stack.shape[0]
    x = np.asarray(stack, np.float32)[: T // m * m]
    blocks = x.reshape(-1, m, *x.shape[1:]).mean(1)
    tau = (m * dt / 2) * blocks.var(0) / x.var(0)
    return np.log(10) * tau


def lifetime_from_stack(stack, method="acf", dt=DIT_10MS, thresh=0.1, **kwargs):
    """Lifetime map by name: 'acf' | 'half' | 'lag1' | 'block' (kwargs: m for 'block')."""
    if method == "acf":
        return crossing_time(temporal_acf(stack), thresh=thresh, dt=dt)
    if method == "half":
        return lifetime_map(stack, dt=dt, thresh=thresh)
    if method == "lag1":
        return t90_lag1(stack, dt=dt)
    if method == "block":
        return t90_block_variance(stack, dt=dt, **kwargs)
    raise ValueError(f"Unknown method {method!r}")

### 2.5 Corner Comparison

In [ ]:
# ------------------------------------------------- the same points for both
def corner_points(Ny, Nx, n, size, seed):
    rng = np.random.default_rng(seed)
    corners = [(0, 0), (0, Nx - size), (Ny - size, 0), (Ny - size, Nx - size)]
    pts = []
    for i in range(n):  # round-robin over the four corners
        y0, x0 = corners[i % 4]
        pts.append((y0 + rng.integers(0, size), x0 + rng.integers(0, size)))
    return np.array(pts)
 
 
# ------------------------------------------------------------------- analysis
def analyse(stack, pts, stop=None):
    stack = stack[:stop]
    N = stack.shape[0]
    series = stack[:, pts[:, 0], pts[:, 1]].astype(float)  # (N, n_points)
    if subtract_mean:
        series = series - series.mean(axis=0, keepdims=True)
 
    X = np.fft.rfft(series, axis=0)
    P = np.abs(X) ** 2
    freq = np.fft.rfftfreq(N, d=dt)
 
    psd = (dt / N) * P
    if N > 2:
        psd[1:-1] *= 2
 
    acf = np.fft.irfft(P, n=N, axis=0)[:n_lags]  # NOT normalised
    return np.mean(stack, axis=0), freq, psd, acf

### 2.5 Plot helpers

Each helper draws one figure and shows it. Where useful they return the computed data (lifetime maps, fit table).

In [ ]:
def plot_region_masks(image, masks, titles=REGION_TITLES):
    """Show `image` restricted to each region mask (2 x 2 grid)."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, (name, region) in zip(axes.ravel(), masks.items()):
        im = ax.imshow(np.where(region, image, np.nan), origin="lower")
        fig.colorbar(im, ax=ax)
        ax.set_title(titles.get(name, name))
    fig.tight_layout()
    plt.show()


def plot_batch_comparison(stack_1, stack_2, profiles=None, title=None,
                          labels=("Batch 1", "Batch 2"), xlabel="Pixel radius"):
    """Mean images (log10) of two stacks, their difference and optionally raw-contrast profiles.

    profiles : ((x1, y1), (x2, y2)), e.g. from `read_raw_contrast`."""
    mean_1, mean_2 = np.mean(stack_1, axis=0), np.mean(stack_2, axis=0)
    n = 4 if profiles else 3
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 4))
    for ax, img, label in zip(axes[:2], (mean_1, mean_2), labels):
        im = ax.imshow(np.log10(img))
        fig.colorbar(im, ax=ax)
        ax.set_title(label, fontsize=14)
    im = axes[2].imshow(mean_2 - mean_1)
    fig.colorbar(im, ax=axes[2])
    axes[2].set_title("Difference", fontsize=14)
    if profiles:
        (x1, y1), (x2, y2) = profiles
        axes[3].plot(x1, y1, label=labels[0], color="k", ls="-")
        axes[3].plot(x2, y2, label=labels[1], color="red", ls=":")
        axes[3].set_yscale("log")
        axes[3].set_xlabel(xlabel)
        axes[3].set_ylabel("Raw contrast")
        axes[3].legend()
    if title:
        fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


def show_raw_pairs(img_files, n_frames=9986):
    """Compare consecutive pairs of raw .tif science files (files 0/1, 2/3, ...)."""
    for f1, f2 in zip(img_files[0::2], img_files[1::2]):
        plot_batch_comparison(
            read_tif_stack(f1, n_frames), read_tif_stack(f2, n_frames),
            title=f"{titling(Path(f1).name)}  vs  {titling(Path(f2).name)}",
        )


def compare_batches(folder, h5_path, ros, wss, algos=ALGOS):
    """Batch 1 vs batch 2 (mean image, difference, raw contrast) for every algo / ro / ws."""
    with pd.HDFStore(h5_path) as store:
        for algo in algos:
            for run in iter_runs(ros, wss):
                ro, ws = run["ro"], run["ws"]
                stacks = [load_stack(folder, algo, ro=ro, ws=ws, count=c) for c in (1, 2)]
                profiles = [read_raw_contrast(store, f"{algo}_{run_name(ro, ws, count=c)}") for c in (1, 2)]
                plot_batch_comparison(*stacks, profiles=profiles,
                                      title=titling(f"{algo}_{ro}ro_{ws}ws"))


def show_outlier_summary(store, png_dir, tag):
    """Speckle-diversity PNGs of both batches plus their mean outlier images.

    store : open h5py.File (outlier_diff.h5); tag : '<algo>_<ro>ro_<ws>ws'."""
    fig = plt.figure(figsize=(19, 8))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.6, 1.6, 0.6], height_ratios=[1, 1],
                          wspace=0.01, hspace=0.1)
    for col, batch in enumerate((1, 2)):
        ax = fig.add_subplot(gs[:, col])
        ax.imshow(plt.imread(Path(png_dir) / f"{tag}_{batch}.png"))
        ax.axis("off")
        ax = fig.add_subplot(gs[col, 2])
        im = ax.imshow(store[f"/{tag}_{batch}"])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(f"Batch {batch} Mean Outlier", fontsize=14)
        ax.axis("off")
    fig.suptitle(titling(tag), fontsize=20)
    plt.show()


def plot_correlation_summary(res, dt, title=None, region_image=None, xlim=(0, 1)):
    """Four panels for one region of `temporal_correlation`: region image (optional),
    correlation vs lag with mean / median / exponential fit, correlation matrix, histogram."""
    cmat = res["correlation_matrix"]
    values = cmat[~np.isnan(cmat)]
    median_corr = res["correlation_median"]
    mean, std = res["correlation_1d_mean"], res["correlation_1d_std"]
    lags = np.arange(1, cmat.shape[0]) * dt

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    ax_img, ax_lag, ax_mat, ax_hist = axes.ravel()

    if region_image is not None:
        im = ax_img.imshow(region_image, origin="lower")
        fig.colorbar(im, ax=ax_img)
    else:
        ax_img.axis("off")

    ax_lag.axhline(median_corr, color="red", ls="--", label=f"Overall median = {median_corr:.3f}")
    ax_lag.plot(lags, mean, color="blue", label="Temporal mean")
    ax_lag.fill_between(lags, mean - std, mean + std, color="blue", alpha=0.1)
    ax_lag.plot(lags, res["correlation_1d_median"], color="red", label="Temporal median")
    if "exp_fit_params" in res:
        A, xi, C0 = res["exp_fit_params"]
        A_s, xi_s, C0_s = res["exp_fit_std"]
        ax_lag.plot(
            lags, exp_decay(lags, A, xi * dt, C0), color="black", ls="--",
            label=f"Exp decay fit\n({A:.2f} ± {A_s:.2f}) exp(-t / ({xi * dt:.3g} ± {xi_s * dt:.2g}) s)"
                  f"\n+ {C0:.1e} ± {C0_s:.1e}",
        )
    ax_lag.set_xlabel("Elapsed time [s]")
    ax_lag.set_ylabel("Correlation coefficient")
    ax_lag.set_xlim(*xlim)
    ax_lag.grid()
    ax_lag.legend(fontsize=8)

    im = ax_mat.imshow(cmat, cmap="viridis")
    fig.colorbar(im, ax=ax_mat, label="correlation")

    ax_hist.hist(values, bins=100)
    ax_hist.axvline(median_corr, color="red", ls="--", label=f"median = {median_corr:.3f}")
    ax_hist.set_yscale("log")
    ax_hist.set_xlabel("correlation coefficient")
    ax_hist.set_ylabel("frequency")
    ax_hist.legend()

    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def plot_grid_correlation(results_grid, dt, xlim=(0, 1), ylim=(-0.1, 1.1)):
    """One small panel per grid tile: mean +/- std, median, overall median and exponential fit.

    Returns a DataFrame (row, col, A, tau [s], C0) with the fit parameters of every tile."""
    pattern = re.compile(r"y(\d+):(\d+)_x(\d+):(\d+)")
    tiles = {}
    for name, res in results_grid.items():
        m = pattern.fullmatch(name)
        if m:
            y0, y1, x0, x1 = map(int, m.groups())
            tiles[name] = (y0, x0, y1 - y0, x1 - x0, res)
    if not tiles:
        raise ValueError("No grid tiles (names like 'y0:12_x0:12') found in results_grid")

    h = max(t[2] for t in tiles.values())
    w = max(t[3] for t in tiles.values())
    n_rows = max(t[0] // h for t in tiles.values()) + 1
    n_cols = max(t[1] // w for t in tiles.values()) + 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.6 * n_cols, 1.6 * n_rows),
                             sharex=True, sharey=True, squeeze=False)
    used, records = set(), []
    for y0, x0, _, _, res in tiles.values():
        row, col = y0 // h, x0 // w
        ax = axes[row, col]
        used.add((row, col))
        mean, std = res["correlation_1d_mean"], res["correlation_1d_std"]
        lags = np.arange(1, len(mean) + 1) * dt
        ax.axhline(res["correlation_median"], color="red", ls="--", lw=0.8)
        ax.plot(lags, mean, color="blue", lw=1)
        ax.fill_between(lags, mean - std, mean + std, color="blue", alpha=0.1)
        ax.plot(lags, res["correlation_1d_median"], color="red", lw=1)
        if "exp_fit_params" in res:
            A, xi, C0 = res["exp_fit_params"]
            ax.plot(lags, exp_decay(lags, A, xi * dt, C0), color="black", ls="--", lw=1)
            records.append(dict(row=row, col=col, A=A, tau=xi * dt, C0=C0))
        ax.grid()
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_title(f"{row}, {col}", fontsize=8)
    for (row, col), ax in np.ndenumerate(axes):
        if (row, col) not in used:
            ax.axis("off")
    for ax in axes[-1, :]:
        ax.set_xlabel("Elapsed time [s]")
    for ax in axes[:, 0]:
        ax.set_ylabel("Correlation")
    fig.tight_layout()
    plt.show()
    return pd.DataFrame(records)


def plot_fit_parameters(fit_df):
    """Exponential-fit amplitude, baseline and decay time of every grid tile."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    for ax, (col, label) in zip(axes, [("A", "Amplitude A"), ("C0", "Baseline C0"), ("tau", "Decay time [s]")]):
        ax.scatter(np.arange(len(fit_df)), fit_df[col], 8)
        ax.set_xlabel("Tile index")
        ax.set_ylabel(label)
        ax.grid()
    fig.tight_layout()
    plt.show()


def plot_pixel_acfs(acf, pixels, lifetime=None, thresh=0.5, max_lag=20):
    """ACF versus lag [samples] of selected pixels (y, x); lifetime map optional for the labels."""
    fig, ax = plt.subplots()
    for y, x in pixels:
        label = f"Pixel ({y}, {x})"
        if lifetime is not None:
            label += f", t = {lifetime[y, x]:.3f} s"
        ax.plot(acf[:, y, x], marker=".", label=label)
    ax.axhline(thresh, color="red", ls="--", label=f"{thresh} threshold")
    ax.set_xlim(0, max_lag)
    ax.set_xlabel("Lag [samples]")
    ax.set_ylabel("Autocorrelation")
    ax.grid()
    ax.legend()
    plt.show()


def plot_lifetime_maps(maps, vmin=5e-3, vmax=1.0, aperture=None):
    """Lifetime maps side by side on a log colour scale; `maps` = {title: 2D array [s]}."""
    fig, axes = plt.subplots(1, len(maps), figsize=(5 * len(maps), 4.5), squeeze=False)
    for ax, (title, lifetime) in zip(axes[0], maps.items()):
        im = ax.imshow(lifetime, origin="lower", cmap="coolwarm", norm=LogNorm(vmin=vmin, vmax=vmax))
        if aperture is not None:
            aperture.plot(ax=ax, color="k", lw=1.5, ls="--")
        fig.colorbar(im, ax=ax).set_label("Lifetime (s)")
        ax.set_xlabel("Pixel X")
        ax.set_ylabel("Pixel Y")
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    plt.show()


def plot_lifetime_comparison(lifetimes, title="Temporal Half-Correlation Lifetime", map_titles=None,
                             colors=None, vmin=5e-3, vmax=1.0, aperture=MASK_APERTURE):
    """Lifetime maps (top, analysis aperture outlined) and ONE shared histogram of the values
    inside the aperture (bottom).

    lifetimes : {label: 2D lifetime map [s]}."""
    colors = colors or {}
    mask = aperture.to_mask(method="center")
    inside = {}
    for label, lifetime in lifetimes.items():
        values = mask.get_values(lifetime)
        inside[label] = values[np.isfinite(values)]

    fig = plt.figure(figsize=(5 * len(lifetimes), 8))
    gs = fig.add_gridspec(2, len(lifetimes), height_ratios=[1.3, 1])
    for i, (label, lifetime) in enumerate(lifetimes.items()):
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(lifetime, origin="lower", cmap="coolwarm", norm=LogNorm(vmin=vmin, vmax=vmax))
        aperture.plot(ax=ax, color="k", lw=1.5, ls="--")
        fig.colorbar(im, ax=ax).set_label("Lifetime (s)")
        ax.set_xlabel("Pixel X")
        ax.set_ylabel("Pixel Y")
        ax.set_title((map_titles or {}).get(label, label))

    ax_h = fig.add_subplot(gs[1, :])
    all_values = np.concatenate(list(inside.values()))
    if all_values.size:
        bins = np.linspace(all_values.min(), all_values.max(), 100)    # shared bins
        for label, values in inside.items():
            ax_h.hist(values, bins=bins, histtype="stepfilled", alpha=0.45, color=colors.get(label),
                      label=f"{label} (median {np.median(values):.3g} s)")
        ax_h.legend()
    ax_h.set_xscale("log")
    ax_h.set_yscale("log")
    ax_h.set_xlabel("Lifetime (s)")
    ax_h.set_ylabel("Counts")
    ax_h.set_title("Lifetime histogram (inside mask)")
    ax_h.grid()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def compare_runs(folder, runs, dt, method="acf", thresh=0.1, algos=ALGOS,
                 vmin=5e-3, vmax=1.0, **method_kwargs):
    """Integrator-vs-predictive lifetime comparison for each run in `runs` (see `iter_runs`).

    method / thresh / method_kwargs are passed to `lifetime_from_stack`.
    Returns {run tag: {label: lifetime map}}."""
    colors = {label: color for label, color in algos.values()}
    out = {}
    for run in runs:
        tag = run_name(**run)
        extra = " ".join(str(run[k]) for k in ("key", "count") if run[k] is not None)
        lifetimes, map_titles = {}, {}
        for algo, (label, _) in algos.items():
            stack = load_stack(folder, algo, **run)
            lifetimes[label] = lifetime_from_stack(stack, method=method, dt=dt, thresh=thresh, **method_kwargs)
            map_titles[label] = titling(f"{algo}_{run['ro']}ro_{run['ws']}ws") + (f"\n{extra}" if extra else "")
        plot_lifetime_comparison(lifetimes, title=f"Lifetime ({method}) - {tag}", map_titles=map_titles,
                                 colors=colors, vmin=vmin, vmax=vmax)
        out[tag] = lifetimes
    return out


def load_temporal_correlation(path, pattern):
    """Concatenate all entries of an HDF store whose key contains `pattern`."""
    dfs = []
    with pd.HDFStore(path, mode="r") as store:
        for key in store.keys():
            if pattern in key:
                print(key)
                dfs.append(store[key])
    return pd.concat(dfs, ignore_index=True)

## 3. Data inspection

### 3.1 Raw frames

Consecutive pairs of raw science files: mean image of each (log10) and their difference.

In [ ]:
img_files, psf_files = list_raw_files(RAW_TIF_DIR)
print(f"{len(img_files)} science files, {len(psf_files)} PSF files")
show_raw_pairs(img_files, n_frames=9986)

### 3.2 Batch 1 vs batch 2 (10 ms data)

For every `algo / ro / ws`: mean image of each batch, their difference, and the raw-contrast profiles from `raw_contrast.h5`. Large differences between the two batches show how reproducible a measurement is.

In [ ]:
compare_batches(RUNS_10_15, RUNS_10_15 / "raw_contrast.h5", ROS, WSS)

### 3.3 Outlier frames

Speckle-diversity plots from the PCA outlier detection of the pipeline (PNG) next to the mean outlier image (`outlier_diff.h5`).

In [ ]:
with h5py.File(RUNS_10_15 / "outlier_diff.h5", "r") as store:
    print(list(store.keys()))
    for algo in ALGOS:
        for run in iter_runs(ROS, WSS):
            show_outlier_summary(store, SPECKLE_PNG_DIR, f"{algo}_{run['ro']}ro_{run['ws']}ws")
            

## 4. Regions and masks

Example stack for sections 4 and 5: predictive controller, 10 ro, 15 ws, batch 2 (10 ms frames).

In [ ]:
DT = DIT_10MS
sci_img = load_stack(RUNS_10_15, "pred", ro=10, ws=15, count=2)
mean_img = sci_img.mean(axis=0)
print(sci_img.shape)

plt.imshow(np.log10(mean_img), origin="lower")
plt.colorbar(label="log10(mean intensity)")
plt.show()

The image is split into the coronagraph, two control stages and the uncontrolled outer region around the image centre (radii in `REGION_EDGES`, in pixels). Neighbouring regions overlap by 2 px.

In [ ]:
center = center_subpixel(mean_img)
region_masks = circular_region_masks(sci_img.shape[-2:], center, **REGION_EDGES)
plot_region_masks(mean_img, region_masks)

## 5. Frame-to-frame correlation matrices

For a set of pixels (a region) every frame is a vector; the correlation between frames $i$ and $j$ is plotted against the lag $|i-j|$. A fast decay means a quickly changing speckle pattern.

### 5.1 Whole image

Without and with subtraction of each pixel's temporal mean. Subtracting the mean removes the static speckle pattern, which otherwise keeps all frames correlated.

In [ ]:
for sub in (None, "mean"):
    res = temporal_correlation(sci_img, sub=sub, fit=False)["full"]
    plot_correlation_summary(res, DT, region_image=mean_img,  title=f"Whole image, sub={sub}")

### 5.2 Control regions

Median-subtracted correlation per region with an exponential fit $A\,e^{-t/\xi}+C_0$ of the mean lag profile (first `t_max` lags).

In [ ]:
results = temporal_correlation(sci_img, masks=region_masks, sub="median", fit_sigma=False)

for name in region_masks:
    plot_correlation_summary(results[name], DT, title=f"{name} mask",
                             region_image=np.where(region_masks[name], mean_img, np.nan))

### 5.3 Spatial grid

The same analysis on a regular grid (12 x 12 px tiles) shows how the decorrelation time varies over the image. Each panel is one tile; the table collects the fit parameters.

In [ ]:
results_grid = temporal_correlation(sci_img, grid_size=12, t_max=100)
fit_df = plot_grid_correlation(results_grid, DT)
fit_df.describe()

In [ ]:
plot_fit_parameters(fit_df)

## 6. Lifetime estimation

Goal: one decorrelation time per pixel from its time trace $I(t)$, as an image in seconds. The long, stacked run (`int`, 7 ro, 20 ws, key 1key, batches 1-4, DIT = 2.814 ms) is used for the walk-through.

In [ ]:
walk_stack = load_stack(LONG_RUN, "int", ro=7, ws=20, key="1key", count="1-4")
walk_dt = DIT_LONG
N, Ny, Nx = walk_stack.shape
print(f"{N} frames of {Ny} x {Nx} px, dt = {walk_dt * 1e3:.3f} ms, T = {N * walk_dt:.1f} s")

### 6.1 Spectral view: power spectrum and variance

For $N$ frames at sampling time $\Delta t$:

- total duration $T = N\,\Delta t$
- frequency resolution $\Delta f = 1/T$
- Nyquist frequency $f_\mathrm{Nyq} = 1/(2\Delta t)$

Example: 10 ms and 10 000 frames give $T = 100\,\mathrm{s}$, $\Delta f = 0.01\,\mathrm{Hz}$ and $f_\mathrm{Nyq} = 50\,\mathrm{Hz}$. At 2.814 ms the Nyquist frequency is about 178 Hz.

The total temporal variance of a pixel is the integral of its PSD, $\sigma^2 = \int_0^{f_\mathrm{Nyq}} P(f)\,df$. The zero-frequency bin is the temporal mean (DC component) and is excluded.

In [ ]:
freq, psd = temporal_psd(walk_stack, walk_dt)
print(f"freq {freq.shape}: {freq[0]:.2f} .. {freq[-1]:.1f} Hz in steps of {freq[1] - freq[0]:.4f} Hz")
print(f"psd  {psd.shape}")

In [ ]:
py, px = 40, 40
total_variance, f_half = half_variance_frequency(freq, psd[:, py, px])

plt.loglog(freq[1:], psd[1:, py, px])           # DC bin excluded
plt.xlabel("Frequency [Hz]")
plt.ylabel("PSD [intensity$^2$/Hz]")
plt.title(f"Pixel ({py}, {px})")
plt.show()

print(f"Total variance: {total_variance:.2e}")
print(f"Half-variance frequency: {f_half:.2f} Hz")

### 6.2 Half-correlation time via the autocorrelation

A frequency is not a time lag. For a stationary signal the PSD and the autocorrelation are a Fourier pair (Wiener-Khinchin theorem). With the mean-subtracted trace $I(t)$,

$$C(\tau) = \langle I(t)\,I(t+\tau)\rangle,\qquad C(\tau_{1/2}) = \tfrac12\,C(0)$$

defines the half-correlation time $\tau_{1/2}$. `wiener_khinchin_acf` returns the ACF normalised to $C(0)=1$; `lifetime_map` wraps these steps and interpolates the first crossing of 0.5.

In [ ]:
acf = wiener_khinchin_acf(walk_stack)
lifetime_half = crossing_time(acf[: N // 2], thresh=0.5, dt=walk_dt)   # same as lifetime_map(walk_stack, walk_dt)

plot_pixel_acfs(acf, [(40, 40), (20, 20), (10, 10), (100, 100)], lifetime=lifetime_half, thresh=0.5)
plot_lifetime_maps({"Half-correlation lifetime": lifetime_half}, vmin=6e-3, vmax=1.0)

### 6.3 ACF threshold crossing (zero-padded)

`temporal_acf` pads each trace to at least $2T-1$ samples before the FFT. Without the padding, lag $k$ wraps the end of the trace onto its start and the ACF decays artificially fast. With `unbiased=True` lag $k$ is divided by its number of overlapping pairs, $T-k$.

`crossing_time` finds the first lag below a threshold and interpolates linearly between the neighbouring frames, so the map is not quantised in steps of `dt`. A threshold of 0.1 gives the $t_{90}$.

In [ ]:
increase_lag = {
    "ACF crossing 0.1 (t90)": lambda s: crossing_time(temporal_acf(s), thresh=0.1, dt=walk_dt),
    "ACF crossing 0.2 (t80)": lambda s: crossing_time(temporal_acf(s), thresh=0.2, dt=walk_dt),
    "ACF crossing 0.3 (t70)": lambda s: crossing_time(temporal_acf(s), thresh=0.3, dt=walk_dt),
    "ACF crossing 0.4 (t60)": lambda s: crossing_time(temporal_acf(s), thresh=0.4, dt=walk_dt),
    "ACF crossing 0.5 (t50)": lambda s: crossing_time(temporal_acf(s), thresh=0.5, dt=walk_dt),
}
increase_lag_maps = {name: fn(walk_stack[:, 20:90, 20:90]) for name, fn in increase_lag.items()}
plot_lifetime_maps(increase_lag_maps, vmin=None, vmax=None)

raw_img = read_tif_stack('/home/aosimul/noah/data/ghost_images/phase_screens_SLM/10_6/int_7ro_20ws_key1_coro_2026-10-06T14-57-05.008_1.tif', n_frames=10000)

increase_lag_raw_maps = {name: fn(raw_img[:, 65:135, 65:135]) for name, fn in increase_lag.items()}
plot_lifetime_maps(increase_lag_raw_maps, vmin=None, vmax=None)

In [ ]:
perfect_img = read_tif_stack('/home/aosimul/noah/data/ghost_images/phase_screens_SLM/8_24/psf_unaberrated_2026-08-25T16-06-03.231_0.tif', n_frames=10000) 
perfect_stack =np.where(circular_region_masks(perfect_img.shape[-2:], (102,99), edge_coro=10, edge_stg1=34, edge_stg2=60,
                          edge_atm=160, overlap=2)['coro'], perfect_img, np.nan)
increase_lag_perfect_maps = {name: fn(perfect_stack[:, 65:135, 65:135]) for name, fn in increase_lag.items()}
plot_lifetime_maps(increase_lag_perfect_maps, vmin=None, vmax=None)

In [ ]:
walk_stack_pred = load_stack(LONG_RUN, "pred", ro=7, ws=20, key="1key", count="1-4")

increase_lag_maps = {name: fn(walk_stack_pred[:, 20:90, 20:90]) for name, fn in increase_lag.items()}
plot_lifetime_maps(increase_lag_maps, vmin=None, vmax=None)

raw_img = read_tif_stack('/home/aosimul/noah/data/ghost_images/phase_screens_SLM/10_6/pred_7ro_20ws_key1_coro_2026-10-06T14-45-58.218_0.tif', n_frames=10000)

increase_lag_raw_maps = {name: fn(raw_img[:, 65:135, 65:135]) for name, fn in increase_lag.items()}
plot_lifetime_maps(increase_lag_raw_maps, vmin=None, vmax=None)

### 6.4 Lag-1 estimator (von Neumann / Durbin-Watson ratio)

If the decay is approximately exponential, one lag determines the whole curve. This is the cheapest estimator: one difference and two sums, no lag loop and no threshold crossing.

$$\mathrm{DW}=\frac{\sum_t (x_{t+1}-x_t)^2}{\sum_t (x_t-\bar x)^2}\approx 2\,(1-r_1),\qquad \tau=-\frac{\Delta t}{\ln r_1},\qquad t_{90}=\ln(10)\,\tau$$

Every sample pair contributes, so this is the best-conditioned estimator for short stacks (a crossing at lag 60 of 100 frames rests on only 40 pairs).

The cost is that it commits to a single exponential: a white-noise floor pulls $r_1$ down and biases $\tau$ short, and a pixel with two timescales is described by the fast one. Computing $r_2$ with `x[:-2], x[2:]` and checking $r_2\approx r_1^2$ tests the model.

### 6.5 Block-variance estimator

No lags and no model of the ACF shape. Average each trace in blocks of $m$ frames and look at how much variance survives. For independent samples the variance of block means falls as $1/m$; for correlated samples it falls more slowly, and the shortfall is the correlation time:

$$\tau \approx \frac{m\,\Delta t}{2}\,\frac{\mathrm{Var}(\text{block means})}{\mathrm{Var}(x)},\qquad t_{90}=\ln(10)\,\tau$$

Sweep $m$ (for example 5, 10, 20) and look for a plateau: that signals the true correlation time has been resolved rather than a partial one. The estimate is too low whenever $m\,\Delta t$ is not comfortably longer than $\tau$, and with 100 frames $m=10$ leaves only 10 blocks, so treat it as a sanity check on the lag-1 map. Its strength is insensitivity to the functional form of the ACF.

### 6.6 Estimators side by side

In [ ]:
estimators = {
    "Half-correlation (0.5)": lambda s: lifetime_map(s, dt=walk_dt, thresh=0.5),
    "ACF crossing (0.1)": lambda s: lifetime_from_stack(s, "acf", dt=walk_dt, thresh=0.1),
    "Lag-1 (t90)": lambda s: t90_lag1(s, dt=walk_dt),
    "Block variance m=10 (t90)": lambda s: t90_block_variance(s, m=10, dt=walk_dt),
}
estimator_maps = {name: fn(walk_stack) for name, fn in estimators.items()}
plot_lifetime_maps(estimator_maps, vmin=2e-3, vmax=1.0, aperture=MASK_APERTURE)

### 6.7 Mean ACF inside the analysis mask

Mean and spread of the temporal ACF of all pixels inside the analysis aperture.

In [ ]:
max_lag = 200
acf_zp = temporal_acf(walk_stack)[:max_lag]
inside = MASK_APERTURE.to_mask(method="center").to_image(walk_stack.shape[1:]) > 0
curves = acf_zp[:, inside]
lags = np.arange(max_lag) * walk_dt

plt.plot(lags, np.nanmean(curves, axis=1), label="mean over pixels")
plt.fill_between(lags, np.nanmean(curves, 1) - np.nanstd(curves, 1),
                 np.nanmean(curves, 1) + np.nanstd(curves, 1), alpha=0.2, label="+/- 1 std")
plt.axhline(0.1, color="red", ls="--", label="0.1 threshold")
plt.xlabel("Lag [s]")
plt.ylabel("Autocorrelation")
plt.grid()
plt.legend()
plt.show()

### 6.8 Block lifetime map from correlation matrices

Coarse map (4 x 4 px blocks): in each block the first lag where the median correlation profile reaches 10 % of the overall median correlation. The correlation matrix is $T \times T$ per block, so this is expensive for very long stacks.

In [ ]:
block_stack = load_stack(RUNS_10_15, "int", ro=7, ws=15, count=1)
block_map = block_lifetime_map(block_stack, grid_size=4, threshold=0.1, dt=DIT_10MS)

plt.imshow(block_map, cmap="coolwarm", origin="lower")
plt.colorbar(label="Lifetime (s)")
plt.title("Block lifetime map (4 x 4 px blocks)")
plt.show()

### 6.9 Check quasi-statics behaviour

1) check lifetime of an uncorrealted PSF
This verifies if there are bench disturbences.

2) check unprocessed stack
This checks if I introduces motion in the pre-processing.


In [ ]:
key = 9986
perfect_img = tiff.imread('/home/aosimul/noah/data/ghost_images/phase_screens_SLM/8_24/psf_unaberrated_2026-08-25T16-06-03.231_0.tif', key=slice(0, key))

plot_lifetime_maps({"ACF crossing 0.1 (t90)": crossing_time(temporal_acf(perfect_img[:, 50:150, 50:150]), thresh=0.1, dt=walk_dt)}, vmin = None, vmax = None)

In [ ]:
key = 9986
raw_img = tiff.imread('/home/aosimul/noah/data/ghost_images/phase_screens_SLM/10_6/int_7ro_20ws_key1_coro_2026-10-06T14-57-05.008_1.tif', key=slice(0, key))
walk_stack = load_stack(LONG_RUN, "int", ro=7, ws=20, key="1key", count="1-4")

plot_lifetime_maps({"ACF crossing 0.1 (t90)": crossing_time(temporal_acf(raw_img[:, 50:150, 50:150]), thresh=0.1, dt=walk_dt)}, vmin = None, vmax = None)

## 7. Comparing runs

`compare_runs` loads the integrator and predictive stack of each run, computes the lifetime maps with the chosen estimator and shows them with one shared histogram of the pixels inside the analysis aperture (dashed circle).

- `method`: `"acf"` (crossing), `"half"`, `"lag1"` or `"block"`
- `thresh`: ACF threshold for `"acf"` and `"half"` (0.1 = $t_{90}$)
- `vmin` / `vmax`: colour range of the maps

### 7.1 Long runs (2.814 ms), with and without focus

In [ ]:
long_runs = list(iter_runs(ros=[7], wss=[20], keys=["1key", "24key", "32key"], counts=["1-4", "5-8"]))

for folder, label in [(LONG_RUN_FOCUS, "focus"), (LONG_RUN, "no focus")]:
    print(f"=== {label} ===")
    compare_runs(folder, long_runs, dt=DIT_LONG, method="acf", thresh=0.1)

### 7.2 10 ms runs, ACF crossing

Colour range left automatic (`vmin = vmax = None`).

In [ ]:
runs_10ms = list(iter_runs(ROS, WSS, counts=[1, 2]))
compare_runs(RUNS_10_15, runs_10ms, dt=DIT_10MS, method="acf", thresh=0.15, vmin=None, vmax=None)

### 7.3 10 ms runs, lag-1 estimator

In [ ]:
compare_runs(RUNS_10_15, runs_10ms, dt=DIT_10MS, method="lag1", vmin=5e-3, vmax=1.0)

### 7.4 Comparing Corners

In [ ]:
 
# ------------------------------------------------------------------ settings
paths = {
    "Integrator": LONG_RUN_FOCUS + "/int_7ro_20ws_1key_1-4.npy",
    "Predictive": LONG_RUN_FOCUS + "/pred_7ro_20ws_1key_1-4.npy",
}
dt = 2.814e-3          # integration time [s]
n_lags = 1000          # lags kept for the ACF
xlim_lags = 20         # plotted lag range
n_points = 5
corner_size = 20       # points are drawn from a corner_size x corner_size patch
seed = 0
subtract_mean = False  # True -> ACF becomes autocovariance (DC removed)
stop = 20000
 
height_ratios = [2.2, 1, 1]   # images get more height than PSD / ACF rows
 
psd_eq = (
    r"$S(f)=\frac{\Delta t}{N}\,|X(f)|^2\;(\times 2$ one-sided$)$" "\n"
    r"$X(f)=\sum_{n=0}^{N-1} x_n\,e^{-2\pi i f n \Delta t}$"
)
acf_eq = (
    r"$R(\tau)=\sum_{n} x_n\,x_{n+\tau}=\mathcal{F}^{-1}\{|X(f)|^2\}$" "\n"
    r"(not normalised, circular)"
)
box = dict(boxstyle="round", fc="white", ec="gray", alpha=0.85)
 
 
data = {name: np.load(p) for name, p in paths.items()}
Ny, Nx = next(iter(data.values())).shape[1:]
pts = corner_points(Ny, Nx, n_points, corner_size, seed)
 
results = {name: analyse(s, pts, stop) for name, s in data.items()}
 
# ----------------------------------------------------------------------- plot
fig, axes = plt.subplots(
    3, 3, figsize=(15, 11), constrained_layout=True,
    gridspec_kw={"height_ratios": height_ratios},
)
colors = plt.cm.tab10(np.arange(n_points))
 
vmin = min(r[0].min() for r in results.values())
vmax = max(r[0].max() for r in results.values())
 
for col, (name, (mean_img, freq, psd, acf)) in enumerate(results.items()):
    # 1) mean frame + marked points
    ax = axes[0, col]
    im = ax.imshow(mean_img, vmin=vmin, vmax=vmax)
    for k, (y, x) in enumerate(pts):
        ax.plot(x, y, "o", ms=6, mfc="none", mec=colors[k], mew=1.5)
    ax.set_title(f"{name}: mean frame")
    fig.colorbar(im, ax=ax, fraction=0.046)
 
    # 2) power spectrum (skip DC)
    ax = axes[1, col]
    for k in range(n_points):
        ax.loglog(freq[1:], psd[1:, k], color=colors[k], alpha=0.5, lw=1.5)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("PSD [intensity²/Hz]")
    ax.grid(True, which="both", alpha=0.3)
 
    # 3) autocorrelation (un-normalised)
    ax = axes[2, col]
    lags = np.arange(acf.shape[0])
    for k in range(n_points):
        ax.plot(lags, acf[:, k], color=colors[k], alpha=0.5, lw=1.5, marker=".", ms=3)
    ax.set_xlim(0, xlim_lags)
    ax.set_xlabel("Lag [samples]")
    ax.set_ylabel("Autocorrelation")
    ax.grid(True, alpha=0.3)
 
# same y-range across the first two columns for PSD and ACF
for row in (1, 2):
    lo = min(a.get_ylim()[0] for a in axes[row, :2])
    hi = max(a.get_ylim()[1] for a in axes[row, :2])
    for a in axes[row, :2]:
        a.set_ylim(lo, hi)
 
# ------------------------------------------------ third column: difference
(n1, n2) = list(results.keys())
mean1, freq, psd1, acf1 = results[n1]
mean2, _, psd2, acf2 = results[n2]
dlabel = f"{n1} − {n2}"
 
# 1) difference of mean frames
ax = axes[0, 2]
dmean = mean1 - mean2
lim = np.abs(dmean).max()
im = ax.imshow(dmean, cmap="RdBu_r", vmin=-lim, vmax=lim)
for k, (y, x) in enumerate(pts):
    ax.plot(x, y, "o", ms=6, mfc="none", mec=colors[k], mew=1.5)
ax.set_title(f"Mean frame difference ({dlabel})")
fig.colorbar(im, ax=ax, fraction=0.046)
 
# 2) PSD difference
ax = axes[1, 2]
dpsd = psd1 - psd2
for k in range(n_points):
    ax.plot(freq[1:], dpsd[1:, k], color=colors[k], alpha=0.5, lw=1.5)
dpsd_mean = dpsd[1:].mean(axis=1)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=np.abs(dpsd_mean).max() * 1e-3)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("ΔPSD [intensity²/Hz]")
ax.set_title(f"PSD difference ({dlabel})")
ax.grid(True, which="both", alpha=0.3)
 
# 3) ACF difference
ax = axes[2, 2]
dacf = acf1 - acf2
for k in range(n_points):
    ax.plot(lags, dacf[:, k], color=colors[k], alpha=0.5, lw=1.5, marker=".", ms=3)
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_xlim(0, xlim_lags)
ax.set_xlabel("Lag [samples]")
ax.set_ylabel("ΔAutocorrelation")
ax.set_title(f"ACF difference ({dlabel})")
ax.grid(True, alpha=0.3)
 
# ------------------------------------------------------------- equations
axes[1, 0].text(0.97, 0.97, psd_eq, transform=axes[1, 0].transAxes,
                ha="right", va="top", fontsize=9, bbox=box)
axes[2, 0].text(0.97, 0.97, acf_eq, transform=axes[2, 0].transAxes,
                ha="right", va="top", fontsize=9, bbox=box)
 
plt.savefig("corner_acf_comparison.png", dpi=150)
plt.show()

## 8. Stored results

Full pipeline output (`temporal_correlation.h5`). Focus: 0.7 arcsec seeing and 20 m/s wind speed, i.e. the `7ro_20ws` entries.

In [ ]:
df = load_temporal_correlation(RESULTS_10_15 / "temporal_correlation.h5", pattern="7ro_20ws")
df

## 9. Appendix: archived drafts

Earlier attempts kept for reference. Everything below is commented out and **not** part of the analysis.

### 9.1 Spatial-neighbour correlation lifetime (does not run)

Draft that correlates each pixel with the pixel `d` positions further along the flattened image. It cannot run as written: `corr_d` has `h*w - d` entries while `t_life[:, :-d]` has shape `(h, w - d)`, and it opens a figure for every `d`. Superseded by `block_lifetime_map` (6.8) and the ACF estimators (6.2-6.5).

In [ ]:
# # Remove temporal median
# stack_centered = stack - np.median(stack, axis=0, keepdims=True)
#
# n_frames, height, width = stack_centered.shape
#
# # Normalize temporal traces
# X = stack_centered.reshape(n_frames, -1)
# X -= X.mean(axis=0, keepdims=True)
#
# std = X.std(axis=0, keepdims=True)
# X /= std
#
# # Result
# t_life = np.full((height, width), np.nan)
#
# for d in range(1, width):
#
#     A = X[:, :-d]
#     B = X[:, d:]
#
#     # Pearson correlation at distance d
#     corr_d = np.sum(A * B, axis=0) / (n_frames - 1)
#     plt.plot(corr_d)
#     plt.title(f"Correlation at distance {d}")
#     plt.xlabel("Pixel Index")
#     plt.ylabel("Correlation Coefficient")
#     plt.show()
#
#     # Locations where correlation has dropped below 10%
#     crossed = np.isnan(t_life[:, :-d]) & (corr_d < 0.1)
#
#     t_life[:, :-d][crossed] = d

### 9.2 `plot_acf` of the flattened stack

`stack.flatten()` is time-major, so lag 1 here is the *spatial* neighbour in the same frame, not the next frame. It is not a temporal ACF; use section 6.7 instead.

In [ ]:
# from statsmodels.graphics.tsaplots import plot_acf
#
# stack = stack - np.median(stack, axis=0, keepdims=True)
#
# d = stack[:].flatten()
# plot_acf(d)

### 9.3 Reading raw frames directly

Alternative to `load_stack` when starting from the raw .tif frames.

In [ ]:
# # img_file = '/home/aosimul/noah/data/ghost_images/phase_screens_SLM/8_24/pred_5ro_10ws_coro_2026-08-25T13-48-57.198_0'
# # key = 9986
# # sci_img = tiff.imread(glob.glob(f"{img_file}*.tif"), key=slice(0, key))